# EchoChain Notebook 02: PySpark Medallion Delta Lakehouse Pipeline Walkthrough

This notebook demonstrates the complete Medallion Lakehouse execution pipeline:
1. **Bronze Layer**: Ingestion with metadata (`_ingested_at`, `_source_file`)
2. **Silver Layer**: Cleaning, deduplication, price currency conversion (USD), and condition mapping
3. **Fuzzy SKU Reconciliation**: Title-to-SKU linking engine connecting unstructured listings to ERP SKUs
4. **Gold Aggregations**: Computing circular economy metrics, CO₂ avoided, and component failure indices

In [1]:
import sys
import os
sys.path.append('..')

from pyspark_pipeline.spark_session import get_spark_session
from pyspark_pipeline.bronze_ingestion import ingest_bronze_tables
from pyspark_pipeline.silver_cleaning import clean_silver_tables
from pyspark_pipeline.fuzzy_matching import reconcile_sku_matching
from pyspark_pipeline.gold_metrics import build_gold_metrics

# Initialize Spark Session
spark = get_spark_session('Notebook-Walkthrough')
print('Spark Engine Active:', spark.version)

In [2]:
# Run Pipeline Stages
ingest_bronze_tables(spark, raw_dir='../data/raw', bronze_dir='../data/bronze')
clean_silver_tables(spark, bronze_dir='../data/bronze', silver_dir='../data/silver')
reconcile_sku_matching(spark, silver_dir='../data/silver')
build_gold_metrics(spark, silver_dir='../data/silver', gold_dir='../data/gold')

In [3]:
import pandas as pd
# Display Gold Circularity Table Summary
gold_resale_df = pd.read_csv('../data/gold/gold_circularity_metrics.csv')
gold_resale_df[['SKU', 'Product', 'resale_index', 'circularity_score', 'co2_avoided_tons']].head(10)